# 2. DAG Execution

The **DAG** (Directed Acyclic Graph) is the backbone of MARES. It models sub-tasks as nodes and their dependencies as edges.

Key capabilities:
- Cycle detection (add_edge raises if it would create a cycle)
- Topological sort (Kahn's algorithm)
- Level grouping — independent nodes run in parallel "waves"


In [ ]:
from mares.graph.dag import DAG

# Build a small DAG
dag = DAG()
for n in (1, 2, 3, 4, 5):
    dag.add_node(n, data=f"task-{n}")

dag.add_edge(1, 3)  # 3 depends on 1
dag.add_edge(2, 3)  # 3 depends on 2
dag.add_edge(3, 4)  # 4 depends on 3
dag.add_edge(4, 5)  # 5 depends on 4

print(f"Nodes: {dag.node_count()}, Edges: {dag.edge_count()}")
print(f"Topological order: {dag.topological_sort()}")
print(f"Levels (waves): {dag.levels()}")
# Level 0: [1, 2] — independent, run in parallel
# Level 1: [3]    — waits for 1,2
# Level 2: [4]    — waits for 3
# Level 3: [5]    — waits for 4

## Cycle Detection

The DAG rejects any edge that would create a cycle:

In [ ]:
from mares.utils.exceptions import PlanningError

dag2 = DAG()
for n in (1, 2, 3):
    dag2.add_node(n)

dag2.add_edge(1, 2)
dag2.add_edge(2, 3)

try:
    dag2.add_edge(3, 1)  # Would create 1→2→3→1 cycle
    print("ERROR: cycle not detected!")
except PlanningError as e:
    print(f"Cycle correctly rejected: {e}")

## Dependency Resolver

The `DependencyResolver` converts a DAG into execution waves for the parallel runner.

In [ ]:
from mares.graph.dependency_resolver import DependencyResolver
from mares.models.sub_task import SubTask

# Create sub-tasks with dependencies
sub_tasks = [
    SubTask(id=1, task="Research API design", depends_on=[]),
    SubTask(id=2, task="Research database schema", depends_on=[]),
    SubTask(id=3, task="Write API code", depends_on=[1]),
    SubTask(id=4, task="Write DB code", depends_on=[2]),
    SubTask(id=5, task="Integration test", depends_on=[3, 4]),
]

# Build DAG from sub-tasks
dag3 = DAG()
for st in sub_tasks:
    dag3.add_node(st.id, data=st)
    for dep in st.depends_on:
        dag3.add_edge(dep, st.id)

resolver = DependencyResolver(dag3)
waves = resolver.execution_waves()

for i, wave in enumerate(waves):
    print(f"Wave {i}: {[st.task[:30] for st in wave]}")